# 15 · Grid search (w, α) trên val (ticket #25)

Chọn cấu hình cho bảng kết quả chính (spec §6) với mô hình gu app dùng, chọn như notebook 14: NDCG@10 val cao nhất trong `{model}_params.json` (hiện là **ALS**, vector bài và decay đã chọn ở #14). Notebook in tên mô hình ở dòng đầu.
- Điểm mix theo luật: 21 bộ trọng số w = (BPM, tông, energy), bước 0,2, tổng bằng 1, × α ∈ {0; 0,1; …; 1} = 231 cấu hình.
- Điểm mix học (#23): chỉ chọn α (11 cấu hình).
- Độ chính xác (HitRate@10, NDCG@10, MRR) và độ mượt (chuỗi tham lam 10 bài, #18) trên **cùng 2.000 phiên val** cố định (`random_seed` 42, cùng mẫu đo độ mượt ở #19).
- Quy tắc chọn: tỉ lệ cặp mix được cao nhất mà NDCG@10 giảm ≤ 10% so với α = 1. Nếu chỉ α = 1 đạt ngưỡng, notebook in **CẢNH BÁO**: báo cả nhóm trước khi chỉnh.

Code: `src/eval/grid.py` (có test: độ chính xác vector hóa và chuỗi sinh chung đều trùng với cách tính từng phiên của #19).

**Input** (Add Input), giống notebook 14: `titeohihi/mixnext-splited`, `mixnext-embeddings` (ALS), `mixnext-item2vec`, `mixnext-mix-learned`. Không thêm `mixnext-lastfm` (cũng có `tracks.parquet`, script sẽ báo trùng).

**Output** (`/kaggle/working/out/`): `grid.csv` (242 dòng), `mix_params.json` (định dạng `artifacts/`).

Bật **Internet** để clone repo. Chỉ cần CPU; ước tính 1–2 giờ (in thời gian từng α).

In [ ]:
# Bootstrap: lấy code từ GitHub. Đang làm trên nhánh b; khi #25 đã merge thì đổi thành "main".
BRANCH = "b"
REPO = "/tmp/MusicRecSys"
!rm -rf {REPO} && git clone -q --depth 1 -b {BRANCH} https://github.com/yashita23/MusicRecSys {REPO}
import sys; sys.path.insert(0, REPO)

In [ ]:
from pathlib import Path
from src.eval.grid import main

OUT = Path("/kaggle/working/out")
table = main(["--input", "/kaggle/input", "--out", str(OUT)])   # tìm file theo tên dưới /kaggle/input; báo lỗi nếu thiếu hoặc trùng

In [ ]:
import pandas as pd
pd.set_option("display.width", 200)
grid = pd.read_csv(OUT / "grid.csv")
cols = ["mix", "w_bpm", "w_key", "w_energy", "alpha", "ndcg", "hit_rate", "mrr", "mixable_pct", "key_compatible_pct", "bpm_delta_pct", "energy_gap"]
rule, learned = grid[grid["mix"] == "rule"], grid[grid["mix"] == "learned"]
threshold = 0.9 * rule.loc[rule["alpha"] == 1.0, "ndcg"].iloc[0]
print(f"Ngưỡng NDCG@10 (luật) = 0,9 × NDCG ở α = 1 = {threshold:.4f}")

print("\nCấu hình được chọn:")
print(grid.loc[grid["chosen"], cols].to_string(index=False, float_format=lambda x: f"{x:.4f}"))

print("\nĐạt ngưỡng, top 10 theo tỉ lệ cặp mix được (luật):")
ok = rule[rule["ndcg"] >= threshold - 1e-12].sort_values(["mixable_pct", "alpha"], ascending=False)
print(ok[cols].head(10).to_string(index=False, float_format=lambda x: f"{x:.4f}"))

print("\nMỗi α: bộ w mượt nhất trong các bộ đạt ngưỡng (luật):")
best = ok.sort_values("mixable_pct", ascending=False).groupby("alpha", sort=True).head(1).sort_values("alpha", ascending=False)
print(best[cols].to_string(index=False, float_format=lambda x: f"{x:.4f}"))

print("\nMix học theo α:")
print(learned[cols].to_string(index=False, float_format=lambda x: f"{x:.4f}"))
print("\nmix_params.json:")
print((OUT / "mix_params.json").read_text(encoding="utf-8"))

## Đăng output

1. **Save Version → Save & Run All (Commit)**.
2. Tab Output → **New Dataset** tên `mixnext-grid`, để **Private**, chỉ giữ `out/` (`grid.csv`, `mix_params.json`).
3. Notebook 14 (#20): Add Input thêm `mixnext-grid` rồi chạy lại, để `artifacts/mix_params.json` là cấu hình vừa chọn (app dùng α và w trong file này).
4. Chép `grid.csv` vào `reports/grid.csv` của repo (#25).

## Kết quả (Kaggle, CPU, 29/11/2026, 7.809 giây ≈ 2 giờ 10 phút)

Mô hình gu: **ALS** (chọn như notebook 14). 2.000 phiên val (`random_seed` 42), độ mượt trên 2.000 × 10 = 20.000 cặp. Thời gian mỗi α giảm dần khi α tăng (852–1.072 giây ở α ≤ 0,5, 84 giây ở α = 1): α càng gần 1 thì các cấu hình càng đi chung đường nên dùng chung ứng viên gu. Không có cảnh báo.

NDCG@10 ở α = 1 (chỉ gu) trên mẫu: **0,0949** (cả val: 0,0967, #14) → ngưỡng 0,9 × 0,0949 = **0,0854**.

| Cấu hình | NDCG@10 | HitRate@10 | MRR | Tỉ lệ cặp mix được (%) | Tông tương thích (%) | ΔBPM TB (%) | lệch energy |
|---|---:|---:|---:|---:|---:|---:|---:|
| α = 1 (chỉ gu) | 0,0949 | 0,1630 | 0,0827 | 5,96 | 21,5 | 14,8 | 0,149 |
| **Luật, w 0,4/0,4/0,2, α = 0,6 (được chọn)** | **0,0871** (−8,2%) | 0,1555 | 0,0746 | **22,35** | 47,6 | 9,8 | 0,139 |
| Luật, w 0,6/0,2/0,2, α = 0,6 | 0,0860 | 0,1575 | 0,0724 | 19,95 | 34,2 | 7,7 | 0,136 |
| Luật, w 0,4/0,6/0, α = 0,7 | 0,0896 | 0,1580 | 0,0772 | 18,60 | 46,6 | 11,5 | 0,148 |
| **Mix học, α = 0,2 (được chọn)** | **0,0922** (−2,8%) | 0,1600 | 0,0800 | **7,42** | 23,2 | 13,6 | 0,126 |
| Mix học, α = 0,1 (dưới ngưỡng) | 0,0836 (−11,9%) | 0,1505 | 0,0720 | 9,61 | 26,5 | 12,2 | 0,106 |
| Mix học, α = 0 (chỉ mix) | 0,0239 | 0,0495 | 0,0260 | 25,02 | 37,8 | 7,5 | 0,066 |

- **Luật:** cấu hình chọn trùng trọng số mặc định của #12 (0,4/0,4/0,2), α = 0,6. Tỉ lệ cặp mix được tăng 5,96% → 22,35% (gấp 3,75 lần), tông tương thích 21,5% → 47,6%, ΔBPM 14,8% → 9,8%, đổi lại NDCG@10 giảm 8,2% (trong mức 10%).
- Với mỗi α, bộ w mượt nhất đạt ngưỡng dịch dần từ chỉ BPM/tông (α 0,9–0,7) sang có energy (α ≤ 0,6); ở α 0,5 và 0,4 tỉ lệ cặp mix được tối đa còn đạt ngưỡng (17,2% và 12,7%) thấp hơn α = 0,6, nên α = 0,6 là điểm tốt nhất.
- **Mix học:** chọn α = 0,2 (NDCG giảm 2,8%), nhưng tỉ lệ cặp mix được chỉ 5,96% → 7,42%, vì điểm học không học lại luật DJ (Spearman với luật 0,165, #24/#57); nó kéo lệch energy xuống mạnh hơn (0,149 → 0,126). Ở α 0,6–0,8 mix học còn tăng nhẹ NDCG (0,0958 ở α 0,7 so với 0,0949): tín hiệu chuyển bài học từ playlist giúp đoán bài kế tiếp.
- `mix_params.json`: luật w 0,4/0,4/0,2, α 0,6; mix học α 0,2 → notebook 14 Add Input `mixnext-grid`, app mặc định α = 0,6 (trước đó 0,7 của #19).
- Bảng đầy đủ (242 dòng): `reports/grid.csv`.
